# 🫁 Lung Function from Breathing Sound
### A self-contained Colab on turning a mask microphone into a spirometer
**ACM Summer School on AI for Health — Hands-on Notebook**
*Inspired by the SpiroMask project — prepared by Nipun Batra's group*

---

Breathing is the one vital sign you can literally *hear*. As air rushes in and out of your airways it becomes turbulent and produces a soft rushing sound. The louder and faster that rushing, the more air is moving. **SpiroMask** asks a simple question: if we put a tiny **microphone inside a commodity N95 or cloth mask**, can we recover clinically useful lung-function numbers from that sound alone?

**Why this matters.** Respiratory disease is a massive global burden:
- About **235 million** people live with **asthma**.
- More than **3 million** people die every year from **COPD** (chronic obstructive pulmonary disease).
- Over **90%** of those COPD deaths happen in **low- and middle-income countries**, where clinics and spirometers are scarce.

Monitoring something as simple as **breathing rate at home** can flag a COPD exacerbation *early*, before it becomes an emergency.

## What the clinic measures: the Pulmonary Function Test (PFT)

The gold-standard lung test is **spirometry**, part of a Pulmonary Function Test (PFT). You breathe into a tube and it records two regimes:

1. **Forced breathing** — inhale fully, then blast all the air out as hard and fast as you can. This produces the **Flow–Volume curve** and the headline numbers:
   - **FEV1** — Forced Expiratory Volume in the first **1 second**.
   - **FVC** — Forced Vital Capacity, the total volume you can force out.
   - **FEV1/FVC ratio** — a value **below ≈ 0.7** suggests an **obstructive** pattern (COPD, asthma), because obstructed airways empty slowly.
2. **Normal (tidal) breathing** — quiet, relaxed breathing. From this we read the **Respiratory Rate** (breaths per minute).

## From phone to mask: SpiroSmart → SpiroMask

Earlier work, **SpiroSmart**, used a **phone microphone** held in front of the mouth. It worked, but smartphone hardware varies enormously (microphone heterogeneity), and a hand-held phone is not suited to continuous **tidal** monitoring.

**SpiroMask** instead retrofits a microphone *inside a mask you already wear*. The mask holds the mic at a fixed, controlled distance from the mouth and nose, so it can capture **both** forced manoeuvres **and** continuous tidal breathing throughout the day.

## The three audio features

SpiroMask turns raw microphone audio into lung-function estimates using three classic signal-processing features:
- **Mel-Frequency Cepstral Coefficients (MFCC)** — a compact description of the *timbre* (spectral shape) of the breath sound.
- **Mel-Frequency Energy** — how much acoustic energy sits in each perceptual (mel) frequency band, over time.
- **Hilbert Envelope** — the instantaneous *loudness* outline of the sound, which tracks how fast air is moving. Breathing rate comes almost entirely from this envelope.

Results are checked against **ATS (American Thoracic Society)** acceptability criteria, and SpiroMask reports a respiration-rate error (MAE) of only about **0.36–0.47 breaths/min**.

> ⚠️ **This notebook is an educational demo, not a medical device.** Every signal here is *synthetic*, generated in code so the notebook runs end-to-end with no recording and no upload. Nothing here should be used for any diagnostic or clinical decision.

## What you will build

Using only **numpy / scipy / matplotlib** (no audio libraries, no `librosa`, no upload) you will:
1. **Synthesise breathing audio** at 8 kHz — turbulent noise whose loudness envelope follows the breathing.
2. Extract the **Hilbert envelope** and read the **respiratory rate** from quiet tidal breathing.
3. Implement **MFCC and Mel-Frequency Energy from scratch** and watch them describe a forced breath.
4. Sketch a **Flow–Volume curve** from a forced exhale and read off didactic **FEV1 / FVC** numbers.

The pipeline in one line:
```
microphone audio       →  Hilbert envelope        →  breathing rate
microphone audio       →  MFCC + mel energy       →  spirometry regression (real system)
forced-exhale envelope →  integrate over time     →  Flow–Volume curve  →  FEV1, FVC
```

In [ ]:
# SpiroMask-style breathing-sound demo. Pure numpy / scipy / matplotlib.
# Colab already has all of these, so nothing needs to be installed.
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, hilbert, find_peaks, decimate
from scipy.fft import rfft, dct

# librosa would give us MFCCs in a single call, but we deliberately build the
# features from scratch so the notebook is fully self-contained and offline.
try:
    import librosa  # noqa: F401
    HAVE_LIBROSA = True
except Exception:
    HAVE_LIBROSA = False

np.set_printoptions(suppress=True)
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

FS = 8000  # sampling rate (Hz) for all breathing audio

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print("Running in Colab :", IN_COLAB)
print("librosa available:", HAVE_LIBROSA, "(not required: features built from scratch)")
print("Sampling rate    :", FS, "Hz")

## 1. Synthesising breathing audio

We do not need a real recording to *learn the method*. A breath sound is essentially **turbulent air noise** — broadband hiss — whose **loudness rises and falls** as air moves. So our generator has just two ingredients:

1. **Band-limited noise** (the hiss of turbulent airflow), multiplied by
2. an **amplitude envelope** that follows the breathing pattern.

We make two clips:
- a **tidal** clip — several quiet, periodic breaths at a known respiratory rate, and
- a **forced** clip — one loud blast with a sharp onset and a long decay, mimicking a forced exhale.

In [ ]:
def bandlimited_noise(n, fs, lo, hi, rng):
    # White noise band-passed to (lo, hi) Hz: a stand-in for turbulent airflow.
    x = rng.standard_normal(n)
    nyq = 0.5 * fs
    hi = min(hi, 0.99 * nyq)
    b, a = butter(4, [lo / nyq, hi / nyq], btype="band")
    return filtfilt(b, a, x)


def synth_tidal(rr_bpm=15.0, fs=FS, dur=20.0, jitter=0.18, seed=0):
    # Quiet periodic breaths: one raised-cosine loudness hump per breath cycle.
    rng = np.random.default_rng(seed)
    t = np.arange(0, dur, 1.0 / fs)
    period = 60.0 / rr_bpm
    n_breaths = int(dur // period)
    width = 0.55 * period                       # audible fraction of each cycle
    env = np.zeros_like(t)
    for k in range(n_breaths):
        centre = period * (k + 0.5) + jitter * rng.standard_normal()
        amp = abs(1.0 + 0.15 * rng.standard_normal())   # breath-to-breath change
        hump = np.cos(np.pi * (t - centre) / width)
        hump = np.where(np.abs(t - centre) < width / 2.0, np.clip(hump, 0, None), 0.0)
        env = env + amp * hump
    noise = bandlimited_noise(t.size, fs, 150.0, 2000.0, rng)
    audio = (0.2 + 0.8 * env) * noise * 0.2     # small DC floor + breathing AM
    return t, audio, rr_bpm


def synth_forced(fs=FS, dur=3.0, tau=0.6, onset=0.05, seed=1):
    # One forced exhale: fast onset, exponential-ish decay, much louder.
    rng = np.random.default_rng(seed)
    t = np.arange(0, dur, 1.0 / fs)
    rise = np.clip(t / onset, 0, 1)                       # sharp linear onset
    decay = np.exp(-np.clip(t - onset, 0, None) / tau)    # exponential decay
    env = rise * decay
    noise = bandlimited_noise(t.size, fs, 150.0, 2200.0, rng)
    audio = env * noise
    return t, audio, env

In [ ]:
RR_TRUE = 15.0
t_tidal, tidal, RR_TRUE = synth_tidal(rr_bpm=RR_TRUE, fs=FS, dur=20.0, seed=0)
t_forced, forced, forced_env_true = synth_forced(fs=FS, dur=3.0, tau=0.6, seed=1)

print("Tidal clip :", tidal.size, "samples =", tidal.size / FS, "s, true RR =",
      RR_TRUE, "breaths/min")
print("Forced clip:", forced.size, "samples =", forced.size / FS, "s")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 5))
ax[0].plot(t_tidal, tidal, lw=0.5, color="#4c72b0")
ax[0].set(title="Tidal (normal) breathing audio: quiet periodic bursts",
          xlabel="time (s)", ylabel="amplitude")
ax[1].plot(t_forced, forced, lw=0.5, color="#c44e52")
ax[1].set(title="Forced expiration audio: one loud blast",
          xlabel="time (s)", ylabel="amplitude")
plt.tight_layout(); plt.show()

## 2. The Hilbert envelope = the loudness outline

The raw audio oscillates thousands of times per second; what we care about is its **outline** — how loud it is at each instant. The **analytic signal** from the Hilbert transform gives exactly that: `envelope(t) = |hilbert(x)(t)|`. This single curve is the workhorse of breathing-rate estimation, and we will reuse it later as an airflow proxy for the Flow–Volume curve.

In [ ]:
def amplitude_envelope(x):
    # Magnitude of the analytic signal = instantaneous amplitude (loudness).
    return np.abs(hilbert(x))


def moving_average(x, win):
    # Simple smoothing; win is in samples. Keeps the array length the same.
    win = max(1, int(win))
    k = np.ones(win) / win
    return np.convolve(x, k, mode="same")


env_tidal = amplitude_envelope(tidal)
env_forced = amplitude_envelope(forced)

# Smooth at a breathing time-scale so we see breaths, not the underlying hiss.
env_tidal_s = moving_average(env_tidal, 0.15 * FS)
env_forced_s = moving_average(env_forced, 0.02 * FS)

fig, ax = plt.subplots(2, 1, figsize=(11, 5))
ax[0].plot(t_tidal, tidal, lw=0.4, color="#bbbbbb")
ax[0].plot(t_tidal, env_tidal_s, lw=2.0, color="#1f77b4", label="Hilbert envelope")
ax[0].set(title="Tidal breathing: waveform + envelope", xlabel="time (s)",
          ylabel="amplitude"); ax[0].legend(loc="upper right")
ax[1].plot(t_forced, forced, lw=0.4, color="#bbbbbb")
ax[1].plot(t_forced, env_forced_s, lw=2.0, color="#c44e52", label="Hilbert envelope")
ax[1].set(title="Forced expiration: waveform + envelope", xlabel="time (s)",
          ylabel="amplitude"); ax[1].legend(loc="upper right")
plt.tight_layout(); plt.show()

## 3. Respiratory rate from the tidal envelope

Each breath is one bump in the smoothed envelope, so **counting bumps per minute = respiratory rate**. We find the envelope peaks, take the **median spacing** between them (robust to the odd missed or extra peak), and convert to breaths per minute. Because we injected a *known* rate, we can measure our own error — the same **MAE** idea SpiroMask reports (~0.36–0.47 breaths/min against a reference).

In [ ]:
def estimate_respiratory_rate(audio, fs, smooth_s=0.15, min_breath_s=2.0):
    # Envelope -> smooth -> peak-pick -> breaths/min from median peak spacing.
    env = moving_average(amplitude_envelope(audio), smooth_s * fs)
    min_dist = int(min_breath_s * fs)        # breaths cannot be closer than this
    peaks, _ = find_peaks(env, distance=max(1, min_dist),
                          prominence=0.3 * np.std(env))
    if len(peaks) >= 2:
        ibi = np.diff(peaks) / fs            # seconds between successive breaths
        rr = 60.0 / np.median(ibi)
    else:
        rr = float("nan")
    return rr, env, peaks


rr_est, env_used, peaks = estimate_respiratory_rate(tidal, FS)
mae = abs(rr_est - RR_TRUE)
print("True (injected) respiratory rate :", RR_TRUE, "breaths/min")
print("Estimated respiratory rate       :", round(rr_est, 2), "breaths/min")
print("Detected breaths                 :", len(peaks))
print("Absolute error (this clip)       :", round(mae, 2), "breaths/min")

tt = np.arange(len(env_used)) / FS
plt.figure(figsize=(11, 3.2))
plt.plot(tt, env_used, color="#1f77b4", lw=1.6, label="smoothed envelope")
plt.plot(tt[peaks], env_used[peaks], "rv", ms=9, label="detected breaths")
plt.title("Breath detection on the tidal envelope  ->  " +
          str(round(rr_est, 2)) + " breaths/min")
plt.xlabel("time (s)"); plt.ylabel("envelope"); plt.legend(loc="upper right")
plt.tight_layout(); plt.show()

## 4. MFCC and Mel-Frequency Energy — from scratch

Breathing rate only needed loudness. To estimate *how much* air is moving (the spirometry numbers) we need the **shape** of the sound — its spectrum — because faster, more turbulent airflow shifts and broadens the spectral content. Two standard features capture this:

- **Mel-Frequency Energy** — slice the spectrum into perceptual **mel** bands (finely spaced at low frequencies, coarsely at high) and measure the log energy in each band, frame by frame.
- **MFCC** — take the **Discrete Cosine Transform** of those log mel energies. The DCT decorrelates the bands and packs the spectral-envelope *shape* into the first ~13 numbers (the cepstrum). It is the classic feature for speech, and it works for breath because breath sound is essentially coloured turbulence.

We implement the whole chain — framing, windowing, FFT power spectrum, triangular mel filterbank, log, DCT — with nothing but numpy and scipy.

In [ ]:
def hz_to_mel(f):
    return 2595.0 * np.log10(1.0 + f / 700.0)


def mel_to_hz(m):
    return 700.0 * (10.0 ** (m / 2595.0) - 1.0)


def mel_filterbank(n_filters, n_fft, fs):
    # Triangular mel filters spanning 0..fs/2 over (n_fft//2 + 1) FFT bins.
    low, high = hz_to_mel(0.0), hz_to_mel(fs / 2.0)
    mel_points = np.linspace(low, high, n_filters + 2)
    hz_points = mel_to_hz(mel_points)
    bins = np.floor((n_fft + 1) * hz_points / fs).astype(int)
    fb = np.zeros((n_filters, n_fft // 2 + 1))
    for m in range(1, n_filters + 1):
        left, centre, right = bins[m - 1], bins[m], bins[m + 1]
        for k in range(left, centre):
            fb[m - 1, k] = (k - left) / max(centre - left, 1)
        for k in range(centre, right):
            fb[m - 1, k] = (right - k) / max(right - centre, 1)
    return fb


def frame_signal(x, frame_len, frame_step):
    # Split into overlapping frames (one per row). Lengths in samples.
    if len(x) < frame_len:
        x = np.pad(x, (0, frame_len - len(x)))
    n_frames = 1 + (len(x) - frame_len) // frame_step
    idx = (np.arange(frame_len)[None, :] +
           frame_step * np.arange(n_frames)[:, None])
    return x[idx]


def compute_mfcc(x, fs, n_mfcc=13, n_filters=26, frame_s=0.025,
                 step_s=0.010, n_fft=512):
    # Pre-emphasis boosts high frequencies (standard speech/audio step).
    pre = np.append(x[0], x[1:] - 0.97 * x[:-1])
    frame_len = int(frame_s * fs)
    frame_step = int(step_s * fs)
    frames = frame_signal(pre, frame_len, frame_step)
    frames = frames * np.hamming(frame_len)[None, :]
    mag = np.abs(rfft(frames, n=n_fft, axis=1))
    power = (mag ** 2) / n_fft
    fb = mel_filterbank(n_filters, n_fft, fs)
    mel_energy = np.maximum(power @ fb.T, 1e-10)     # avoid log(0)
    log_mel = np.log(mel_energy)                     # log Mel-Frequency Energy
    mfcc = dct(log_mel, type=2, axis=1, norm="ortho")[:, :n_mfcc]
    times = (np.arange(frames.shape[0]) * frame_step + frame_len / 2.0) / fs
    return mfcc, log_mel, times


mfcc, log_mel, frame_times = compute_mfcc(forced, FS)
print("MFCC matrix shape  (frames x coeffs)   :", mfcc.shape)
print("Log-mel matrix shape (frames x bands)  :", log_mel.shape)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6))

im0 = ax[0].imshow(mfcc.T, origin="lower", aspect="auto", cmap="viridis",
                   extent=[frame_times[0], frame_times[-1], 0, mfcc.shape[1]])
ax[0].grid(False)
ax[0].set(title="MFCC of the forced exhale (first 13 coefficients)",
          xlabel="time (s)", ylabel="MFCC index")
fig.colorbar(im0, ax=ax[0], pad=0.01)

# Mel-Frequency Energy over time: total log energy summed across mel bands.
total_mel_energy = log_mel.sum(axis=1)
ax[1].plot(frame_times, total_mel_energy, color="#c44e52", lw=1.8)
ax[1].set(title="Mel-Frequency Energy over time (sum of log mel bands)",
          xlabel="time (s)", ylabel="log energy")
plt.tight_layout(); plt.show()

The MFCC heatmap shows the spectral envelope **collapsing as the blast decays**: the loud onset frames are spectrally rich, while later frames are quiet and dominated by the low-order coefficients. The mel-energy trace mirrors that loudness decay. Because breath sound is broadband turbulence, its *energy distribution across mel bands* scales with how forcefully air is moving — which is exactly what lets a regression model map these features to spirometry values.

## 5. From a forced exhale to a Flow–Volume curve

In real spirometry the **flow** of air (litres/second) is measured directly. Here we use a didactic proxy: the **envelope of the forced-exhale sound stands in for airflow** — louder hiss means faster air. Integrating flow over time gives the **cumulative exhaled volume**. Plotting **Flow (y) versus Volume (x)** traces the expiratory limb of the classic spirometry **Flow–Volume loop**: a fast rise to peak flow, then a decline as the lungs empty.

From this curve we read:
- **FVC** — total volume exhaled (where the curve reaches the volume axis),
- **FEV1** — volume exhaled in the **first second**,
- **FEV1/FVC** — the diagnostic ratio.

Everything below is a **synthetic, didactic illustration**, scaled to look physiological — it is *not* a real flow measurement.

In [ ]:
# Use the smoothed forced-exhale envelope as an airflow proxy, scaled to a
# plausible peak expiratory flow so the numbers look physiological.
PEAK_FLOW_LPS = 9.0                                  # litres/second, healthy peak
flow = env_forced_s / env_forced_s.max() * PEAK_FLOW_LPS
dt = 1.0 / FS
volume = np.cumsum(flow) * dt                        # litres exhaled so far

FVC = volume[-1]                                     # forced vital capacity (L)
one_sec = int(1.0 * FS)
FEV1 = volume[one_sec] if one_sec < len(volume) else volume[-1]
ratio = FEV1 / FVC

print(f"FVC  (total exhaled volume) : {FVC:.2f} L")
print(f"FEV1 (volume in first 1 s)  : {FEV1:.2f} L")
print(f"FEV1/FVC ratio              : {ratio:.2f}")
interp = "normal" if ratio >= 0.70 else "suggestive of airway obstruction"
print(f"Didactic interpretation     : ratio {ratio:.2f} -> {interp}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(t_forced, flow, color="#c44e52", lw=1.5)
ax[0].axvline(1.0, color="k", ls="--", lw=1, label="1 s mark")
ax[0].set(title="Airflow proxy vs time", xlabel="time (s)",
          ylabel="flow (L/s)"); ax[0].legend()
ax[1].plot(volume, flow, color="#4c72b0", lw=1.8)
ax[1].set(title="Flow-Volume curve (expiratory limb)",
          xlabel="exhaled volume (L)", ylabel="flow (L/s)")
plt.tight_layout(); plt.show()

In [ ]:
# Didactic contrast: an OBSTRUCTED exhale empties more slowly (longer decay).
_, forced_obs, _ = synth_forced(fs=FS, dur=3.0, tau=1.4, seed=2)
env_obs = moving_average(amplitude_envelope(forced_obs), 0.02 * FS)
flow_obs = env_obs / env_obs.max() * PEAK_FLOW_LPS * 0.7   # weaker, slower blast
vol_obs = np.cumsum(flow_obs) * dt
FVC_o = vol_obs[-1]
FEV1_o = vol_obs[one_sec] if one_sec < len(vol_obs) else vol_obs[-1]
ratio_o = FEV1_o / FVC_o

print(f"Healthy    FEV1/FVC = {ratio:.2f}  ({'normal' if ratio >= 0.7 else 'obstructed'})")
print(f"Obstructed FEV1/FVC = {ratio_o:.2f}  ({'normal' if ratio_o >= 0.7 else 'obstructed'})")

plt.figure(figsize=(7, 4.2))
plt.plot(volume, flow, color="#4c72b0", lw=1.8, label=f"healthy (ratio {ratio:.2f})")
plt.plot(vol_obs, flow_obs, color="#c44e52", lw=1.8,
         label=f"obstructed (ratio {ratio_o:.2f})")
plt.title("Flow-Volume curves: healthy vs obstructed (synthetic)")
plt.xlabel("exhaled volume (L)"); plt.ylabel("flow (L/s)")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Real-world messiness: noise, speech, and sampling rate

A mask mic in the wild also records **room noise and the wearer's speech**, which can swamp the quiet breath sound. SpiroMask handles this by (a) band-limiting to the breath-sound frequencies, (b) using the envelope (robust to broadband noise), and (c) detecting and rejecting speech segments before estimating breathing — speech has strong, structured spectral content (formants and pitch harmonics) that looks nothing like the smooth turbulence of breathing.

One practical finding from the project: **lowering the sampling rate barely changed the breathing-rate estimate**. Breath-sound energy and its loudness envelope live at low frequencies, so even after we discard the high-frequency detail, the envelope — and therefore the respiratory rate — is essentially unchanged. That means cheaper, lower-power microphones and less data are perfectly adequate for tidal monitoring.

In [ ]:
# Demonstrate that halving the sampling rate barely moves the RR estimate.
tidal_ds = decimate(tidal, 2)            # 8000 Hz -> 4000 Hz (anti-aliased)
fs_ds = FS // 2
rr_full, _, _ = estimate_respiratory_rate(tidal, FS)
rr_half, _, _ = estimate_respiratory_rate(tidal_ds, fs_ds)
print(f"RR at {FS} Hz : {rr_full:.2f} breaths/min")
print(f"RR at {fs_ds} Hz : {rr_half:.2f} breaths/min")
print(f"Difference     : {abs(rr_full - rr_half):.3f} breaths/min  (negligible)")

## 🔬 Bonus: real SpiroMask data (mask-mic audio + clinical ground truth)

Now the real thing. We download (1) a **forced-breath (FVC) audio clip recorded
through an N95 mask** and (2) the **clinical ground-truth spirometry** for the
same person — both released with SpiroMask
([github.com/AyushShrivstava/SpiroMask_DIY](https://github.com/AyushShrivstava/SpiroMask_DIY)).
We run the *same* Hilbert-envelope / spectrogram / flow–volume pipeline on the
real audio, then compare against that participant's real FEV1 / FVC measured by
a clinical spirometer.

Important honesty note: the microphone gives the **shape** of airflow over time;
absolute litres need calibration — which is exactly why a clinical reference is
used to train and validate SpiroMask. So below, the *curve shape* comes from the
real audio, while the *litre values* come from the clinical ground-truth file.

In [ ]:
import urllib.request, os, wave, csv
AUDIO_URL = "https://raw.githubusercontent.com/AyushShrivstava/SpiroMask_DIY/main/2.Data/2.SpiroMask_Audio_Samples/P0.FVC_N95.LCL.2dbk4ib2.wav"
GT_URL    = "https://raw.githubusercontent.com/AyushShrivstava/SpiroMask_DIY/main/2.Data/1.Ground_Truth/GroundTruth_Dataset.csv"
audio_path = "/tmp/spiromask_fvc.wav"
gt_path    = "/tmp/spiromask_gt.csv"
AUDIO_UID  = "P0"     # the participant id encoded in the audio filename
have_audio = False
try:
    for url, path in [(AUDIO_URL, audio_path), (GT_URL, gt_path)]:
        if not os.path.exists(path) or os.path.getsize(path) < 200:
            urllib.request.urlretrieve(url, path)
    have_audio = os.path.getsize(audio_path) > 1000
    print("Ready:", audio_path, "(%d bytes)," % os.path.getsize(audio_path),
          gt_path, "(%d bytes)" % os.path.getsize(gt_path))
except Exception as e:
    print("Could not fetch real data (offline?). Skipping real-data demo.")
    print(" ", e)

In [ ]:
from scipy.signal import hilbert, butter, filtfilt

if have_audio:
    w = wave.open(audio_path)
    fs_r = w.getframerate()
    raw = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(float)
    w.close()
    x = raw / (np.abs(raw).max() + 1e-9)
    t = np.arange(len(x)) / fs_r
    print("Real mask-mic clip: %.2f s @ %d Hz" % (len(x) / fs_r, fs_r))

    # Hilbert envelope -> smooth -> flow proxy; cumulative volume = integral of flow
    env = np.abs(hilbert(x))
    b, a = butter(3, 6.0 / (fs_r / 2.0)); env_s = filtfilt(b, a, env)
    onset = int(np.argmax(env_s > 0.2 * env_s.max()))
    flow = env_s[onset:]
    vol = np.cumsum(flow) / fs_r
    fvc_proxy = vol[-1]
    one_s = min(fs_r, len(vol) - 1)
    fev1_proxy = vol[one_s]
    ratio_audio = fev1_proxy / fvc_proxy
    print("From the AUDIO shape  : FEV1/FVC proxy = %.2f (uncalibrated, shape only)" % ratio_audio)

    # --- look up this participant's REAL clinical spirometry ---
    rows = list(csv.DictReader(open(gt_path)))
    gt = next((r for r in rows if r["UID"] == AUDIO_UID), None)
    if gt:
        gfev1, gfvc = float(gt["gFEV1"]), float(gt["gFVC"])
        print("Clinical ground truth (%s): FEV1=%.2f L, FVC=%.2f L, FEV1/FVC=%.2f"
              % (AUDIO_UID, gfev1, gfvc, gfev1 / gfvc))
        print("  %s, age %s, lung ailment: %s"
              % (gt["Sex"], gt["Age"], gt["Lung Ailment"]))
        print("  clinical FEV1/FVC %.2f -> %s"
              % (gfev1 / gfvc, "normal (>=0.70)" if gfev1 / gfvc >= 0.70 else "obstructed (<0.70)"))

    # --- plots: real waveform+envelope, real spectrogram, real flow-volume ---
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
    ax[0].plot(t, x, color="#bbb", lw=0.4)
    ax[0].plot(t, env_s, color="#d62728", lw=1.5, label="Hilbert envelope")
    ax[0].set_title("Real N95 mask-mic FVC breath"); ax[0].set_xlabel("time (s)")
    ax[0].legend(loc="upper right")

    win, hop = 512, 256
    nseg = 1 + (len(x) - win) // hop
    S = np.empty((win // 2 + 1, nseg))
    hw = np.hanning(win)
    for i in range(nseg):
        seg = x[i * hop:i * hop + win] * hw
        S[:, i] = np.abs(np.fft.rfft(seg)) ** 2
    ax[1].imshow(10 * np.log10(S + 1e-12), origin="lower", aspect="auto",
                 cmap="magma", extent=[0, len(x) / fs_r, 0, fs_r / 2])
    ax[1].set_title("Real spectrogram (breath turbulence)")
    ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("Hz")

    ax[2].plot(vol, flow, color="#1f77b4")
    ax[2].set_title("Flow-volume shape (from real audio)")
    ax[2].set_xlabel("cumulative volume (a.u.)"); ax[2].set_ylabel("flow (a.u.)")
    plt.tight_layout(); plt.show()
    print("\nSame pipeline as the synthetic demo, now on a real recording. "
          "The audio fixes the curve SHAPE; the litre values come from the clinic.")

## 7. Limitations, exercises, and safety

**Real vs synthetic.** Every signal here was generated in code. Real breath sound is far messier: airway anatomy, mask fit, mic placement, ambient noise and motion all matter. Mapping acoustic features to *absolute* spirometry numbers (FEV1, FVC) requires a **calibrated regression model trained on real spirometer data**, validated against **ATS** acceptability criteria — not the toy integration we did here.

**Why the SpiroMask idea is still powerful.** It turns something people already wear into a continuous lung monitor: tidal breathing rate all day (low error, ~0.36–0.47 breaths/min), plus on-demand forced manoeuvres — most useful exactly where spirometers are scarce.

**Exercises**
1. **Noise robustness.** Add white or babble noise to the tidal clip and plot respiratory-rate error versus signal-to-noise ratio. At what SNR does peak-counting break?
2. **Speech rejection.** Synthesise a short voiced segment (a sum of harmonics with formant-like peaks), splice it into the tidal clip, and design a simple detector (e.g. spectral flatness, or a low MFCC-based score) to mask it out before estimating RR.
3. **Feature sweep.** Vary the number of mel filters (13, 26, 40) and MFCCs kept; see how the MFCC heatmap and any downstream fit change.
4. **Inhale vs exhale.** Extend the tidal generator to give each breath two humps (inhale + exhale) and adapt the peak logic so you still recover the correct breathing rate.

**Safety.** Educational only. Synthetic data, didactic math, no clinical validation. Do **not** use any number from this notebook for diagnosis or treatment.

---
*Made for the ACM Summer School on AI for Health — inspired by SpiroMask.*